# Debug: Gemma 4 prompt + live chain of thought

Runs a single gemma4-e2b mate1-lichess position with:
- the EXACT rendered prompt printed (special tokens visible)
- the model's output **streamed live** as tokens are generated
- the parsed verdict after generation

Watch: does the model answer directly (thinking disabled worked), or does it still emit `<|channel>thought` reasoning and get cut off at the token budget?

## 1. Fresh clone + deps

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "neuro-symbolic-pathfinding"
if REPO.exists():
    shutil.rmtree(REPO)
token = os.environ.get("GITHUB_TOKEN") or os.environ.get("GH_TOKEN") or ""
url = "https://github.com/Vedang-P/neuro-symbolic-pathfinding.git"
if token:
    url = url.replace("https://", f"https://x-access-token:{token}@")
subprocess.run(["git", "clone", "--quiet", url, str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-U", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "--quiet", "-y", "wandb"], check=True)
import transformers
print("transformers", transformers.__version__)

## 2. One gemma cell, fully visible

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/run_chess.py",
                "--model", "gemma4-e2b", "--task", "mate1-lichess",
                "--prompt-variant", "grid", "--n", "1",
                "--max_new_tokens", "1024",
                "--output_dir", "results_debug/chess",
                "--verbose", "--stream"], check=False)

## What to look for
- **Prompt block**: starts `<bos><|turn>user...<turn|>\n<|turn>model\n` — no thinking tokens (`<|think>`, `<|channel>`).
- **Streaming output**: if the first tokens are plain text answering with `MOVE:`, the fix worked. If you see `<|channel>thought` or long reasoning followed by truncation at ~1024 tokens, paste the output here — we'll raise the budget and/or tighten the prompt.